# 03. DDL vs DML

*Module: 1 — Basics · Companion summary: `lectures/03-types-of-sql-statements-ddl-vs-dml.md` · SQL engine: SQLite (in-memory)*

## What you will study

- name the four DDL statements and what each does
- map the four DML statements to CRUD
- distinguish structure changes from data changes in practice

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 📗 The two families

**DDL — Data Definition Language** (touches *structure*): `CREATE`, `ALTER`,
`TRUNCATE`, `DROP`.
**DML — Data Manipulation Language** (touches *content*, the CRUD cycle):
`INSERT` (Create), `SELECT` (Read), `UPDATE` (Update), `DELETE` (Delete).

## 🧪 DDL in action — building the container

In [ ]:
q("""
CREATE TABLE notes (
    id     INTEGER PRIMARY KEY,
    body   TEXT,
    author TEXT
)
""")

## 🧪 DML in action — working with what's inside

In [ ]:
q("INSERT INTO notes VALUES (1, 'hello', 'Raul')")
q("INSERT INTO notes VALUES (2, 'world', 'Rav')")
q("UPDATE notes SET author = 'Ana' WHERE id = 2")
q("SELECT * FROM notes")

## 🧪 DELETE (DML) vs DROP (DDL) on the same table

In [ ]:
# DML: remove rows — table survives, schema survives
q("DELETE FROM notes WHERE id = 1")
q("SELECT * FROM notes")
q("SELECT * FROM sqlite_master WHERE name = 'notes' AND type = 'table'")

In [ ]:
# DDL: remove the whole container
q("DROP TABLE notes")
try:
    conn.execute("SELECT * FROM notes")
except sqlite3.OperationalError as e:
    print('Dropping the table also removed its schema:', repr(e))

## 💡 Tips & tricks

- **DELETE is DML, TRUNCATE is DDL** — a favorite interview question.
- Both `TRUNCATE` and `DELETE` without a WHERE empty a table; TRUNCATE is the faster,
  storage-resetting one — but SQLite only has `DELETE FROM table` (no TRUNCATE).
  📕 Dialect note:
  ```sql
  TRUNCATE TABLE notes IMMEDIATE;   -- DB2 / MySQL: fast empty + storage reset
  DELETE FROM notes;                -- SQLite's spelling
  ```

## 🎯 Your turn

1. Recreate `notes` with an extra column `priority INTEGER`.
2. Empty it two different ways (DELETE; then DROP) and check `sqlite_master` in between.

In [ ]:
# 💡 Solution
q("CREATE TABLE notes (id INTEGER PRIMARY KEY, body TEXT, author TEXT, priority INTEGER)")
q("DELETE FROM notes")
q("SELECT count(*) FROM notes")
q("DROP TABLE notes")
try:
    q("SELECT * FROM notes")
except sqlite3.OperationalError as e:
    print('gone:', e)